# Notebook 12 — ORPO and KTO Alignment

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cyruslayo/castalia/blob/main/finetuning/12_orpo_and_kto_alignment.ipynb)

DPO is the default second stage after SFT, but it is not the only useful option. In this notebook you will compare a tiny DPO baseline against ORPO and KTO style objectives so you can see when reference-free pairwise training or binary-label alignment is the better operational fit.

In [ ]:
# --- Google Colab Pro Fine-Tuning + Evaluation Setup ---
%%capture
!pip install -q --upgrade pip
!pip install -q unsloth datasets trl peft accelerate bitsandbytes pandas matplotlib scikit-learn

import json
import math
import random
import time
from pathlib import Path
from typing import Any, Dict, List

import torch
from datasets import Dataset, DatasetDict, load_dataset
from google.colab import drive
from transformers import TrainingArguments
from trl import SFTTrainer
from unsloth import FastLanguageModel

drive.mount('/content/drive')

CACHE_DIR = "/content/drive/MyDrive/models"
OUTPUT_DIR = "/content/drive/MyDrive/castalia-finetuning"
MODEL_NAME = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048
DTYPE = None
LOAD_IN_4BIT = True

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
    cache_dir=CACHE_DIR,
)

tokenizer.padding_side = "right"

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

def formatting_prompts_func(batch):
    return {"text": batch["text"]}

print("✓ Fine-tuning stack ready")
print("  Model:", MODEL_NAME)
print("  CUDA device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("  BF16 supported:", torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

## What you will build

- a shared toy response bank
- pairwise preference data for DPO and ORPO
- binary desirable and undesirable labels for KTO
- first-principles toy objectives for all three methods
- training logs and held-out comparisons
- a practical decision matrix for DPO, ORPO, and KTO

## Step 1: Install lightweight packages and initialize the notebook

Everything in this notebook is local and open-source. We keep the experiment small so the differences between DPO, ORPO, and KTO come from the objectives rather than from infrastructure complexity.

In [ ]:
import importlib.util
import json
import random
import subprocess
import sys
from pathlib import Path

def ensure_package(package_name, import_name=None):
    import_name = import_name or package_name
    if importlib.util.find_spec(import_name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", package_name])

for package_name, import_name in [
    ("pandas", "pandas"),
    ("matplotlib", "matplotlib"),
    ("torch", "torch"),
]:
    ensure_package(package_name, import_name)

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_colwidth", 120)

random.seed(12)
torch.manual_seed(12)

ARTIFACT_DIR = Path("artifacts") / "notebook_12_orpo_kto"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Artifacts will be written to:", ARTIFACT_DIR.resolve())

## Step 2: Create one shared response bank for all three objectives

Using the same response bank makes the comparison fair. The only thing that changes across runs is the learning objective and the format of the supervision signal.

In [ ]:
feature_names = ["helpfulness", "correctness", "safety", "specificity", "concision"]

response_rows = [
    {
        "response_id": "a01_good",
        "prompt_id": "p01",
        "split": "train",
        "prompt": "Explain why regular backups matter.",
        "variant": "chosen",
        "text": "Backups reduce recovery time because they preserve a restorable copy after deletion, ransomware, or failed updates.",
        "features": [0.86, 0.88, 0.82, 0.78, 0.72],
    },
    {
        "response_id": "a01_bad",
        "prompt_id": "p01",
        "split": "train",
        "prompt": "Explain why regular backups matter.",
        "variant": "rejected",
        "text": "Backups are useful because experts usually recommend them.",
        "features": [0.48, 0.50, 0.82, 0.34, 0.56],
    },
    {
        "response_id": "a02_good",
        "prompt_id": "p02",
        "split": "train",
        "prompt": "Rewrite this update so it sounds calm and clear.",
        "variant": "chosen",
        "text": "The database is currently unavailable, which is delaying some work. We are restoring service now and will share another update soon.",
        "features": [0.82, 0.86, 0.80, 0.76, 0.78],
    },
    {
        "response_id": "a02_bad",
        "prompt_id": "p02",
        "split": "train",
        "prompt": "Rewrite this update so it sounds calm and clear.",
        "variant": "rejected",
        "text": "Everything is delayed again because the database failed and nobody knows when it will be fixed.",
        "features": [0.44, 0.54, 0.70, 0.32, 0.46],
    },
    {
        "response_id": "a03_good",
        "prompt_id": "p03",
        "split": "train",
        "prompt": "Should customer data be copied to a personal USB drive?",
        "variant": "chosen",
        "text": "No. Customer data should remain on approved managed devices, so a personal USB drive is not acceptable.",
        "features": [0.78, 0.90, 0.98, 0.72, 0.76],
    },
    {
        "response_id": "a03_bad",
        "prompt_id": "p03",
        "split": "train",
        "prompt": "Should customer data be copied to a personal USB drive?",
        "variant": "rejected",
        "text": "It is acceptable if the employee plans to delete the files later.",
        "features": [0.28, 0.22, 0.06, 0.50, 0.64],
    },
    {
        "response_id": "a04_good",
        "prompt_id": "p04",
        "split": "train",
        "prompt": "Explain what SELECT name FROM orders WHERE total > 100 returns.",
        "variant": "chosen",
        "text": "It returns the name field for rows in orders where the total value is greater than 100.",
        "features": [0.88, 0.92, 0.78, 0.80, 0.76],
    },
    {
        "response_id": "a04_bad",
        "prompt_id": "p04",
        "split": "train",
        "prompt": "Explain what SELECT name FROM orders WHERE total > 100 returns.",
        "variant": "rejected",
        "text": "It returns every order and changes the total to 100.",
        "features": [0.32, 0.14, 0.78, 0.38, 0.74],
    },
    {
        "response_id": "a05_good",
        "prompt_id": "p05",
        "split": "train",
        "prompt": "Write JSON with keys action, owner, deadline from: Maya will patch the server by Tuesday.",
        "variant": "chosen",
        "text": "{\"action\": \"patch the server\", \"owner\": \"Maya\", \"deadline\": \"Tuesday\"}",
        "features": [0.84, 0.88, 0.80, 0.86, 0.70],
    },
    {
        "response_id": "a05_bad",
        "prompt_id": "p05",
        "split": "train",
        "prompt": "Write JSON with keys action, owner, deadline from: Maya will patch the server by Tuesday.",
        "variant": "rejected",
        "text": "Maya should patch the server by Tuesday.",
        "features": [0.54, 0.58, 0.80, 0.34, 0.54],
    },
    {
        "response_id": "a06_good",
        "prompt_id": "p06",
        "split": "train",
        "prompt": "Summarize recursion for a beginner in three bullet points.",
        "variant": "chosen",
        "text": "- A function can call itself.\n- A base case stops the process.\n- Each call works on a smaller problem.",
        "features": [0.84, 0.82, 0.76, 0.82, 0.70],
    },
    {
        "response_id": "a06_bad",
        "prompt_id": "p06",
        "split": "train",
        "prompt": "Summarize recursion for a beginner in three bullet points.",
        "variant": "rejected",
        "text": "Recursion is a self-referential computational strategy with layered semantics.",
        "features": [0.40, 0.44, 0.76, 0.48, 0.38],
    },
    {
        "response_id": "a07_good",
        "prompt_id": "p07",
        "split": "eval",
        "prompt": "Write a calm reply to a delayed support ticket.",
        "variant": "chosen",
        "text": "Thanks for your patience. I am still investigating the issue and will send the next update by tomorrow morning.",
        "features": [0.82, 0.80, 0.84, 0.72, 0.72],
    },
    {
        "response_id": "a07_bad",
        "prompt_id": "p07",
        "split": "eval",
        "prompt": "Write a calm reply to a delayed support ticket.",
        "variant": "rejected",
        "text": "We are late because other work took priority.",
        "features": [0.48, 0.58, 0.72, 0.38, 0.54],
    },
    {
        "response_id": "a08_good",
        "prompt_id": "p08",
        "split": "eval",
        "prompt": "Extract action items as JSON from: Priya will review the draft by Monday.",
        "variant": "chosen",
        "text": "{\"action\": \"review the draft\", \"owner\": \"Priya\", \"deadline\": \"Monday\"}",
        "features": [0.84, 0.88, 0.80, 0.84, 0.68],
    },
    {
        "response_id": "a08_bad",
        "prompt_id": "p08",
        "split": "eval",
        "prompt": "Extract action items as JSON from: Priya will review the draft by Monday.",
        "variant": "rejected",
        "text": "Priya should review the draft before Monday.",
        "features": [0.52, 0.56, 0.80, 0.36, 0.52],
    },
    {
        "response_id": "solo_pos",
        "prompt_id": "p09",
        "split": "train",
        "prompt": "Politely ask a teammate for an update.",
        "variant": "desirable_only",
        "text": "Hi Sam, when you have a moment, could you share the latest status on the ticket?",
        "features": [0.80, 0.78, 0.84, 0.68, 0.74],
    },
    {
        "response_id": "solo_neg",
        "prompt_id": "p10",
        "split": "train",
        "prompt": "Politely ask a teammate for an update.",
        "variant": "undesirable_only",
        "text": "Why have you still not finished this ticket?",
        "features": [0.34, 0.56, 0.70, 0.42, 0.60],
    },
]

response_df = pd.DataFrame(response_rows)
response_df[["response_id", "prompt_id", "split", "variant"]].head(16)

## Step 3: Build pairwise and binary-label supervision views

DPO and ORPO consume chosen and rejected pairs. KTO works from desirable and undesirable labels on individual responses. The same underlying examples can be viewed both ways, but KTO can also consume standalone thumbs-up and thumbs-down data.

In [ ]:
pair_rows = []
for prompt_id, group in response_df[response_df["variant"].isin(["chosen", "rejected"])].groupby("prompt_id"):
    chosen_row = group[group["variant"] == "chosen"].iloc[0]
    rejected_row = group[group["variant"] == "rejected"].iloc[0]
    pair_rows.append(
        {
            "pair_id": f"{prompt_id}_pair",
            "prompt_id": prompt_id,
            "split": chosen_row["split"],
            "prompt": chosen_row["prompt"],
            "chosen_id": chosen_row["response_id"],
            "rejected_id": rejected_row["response_id"],
            "chosen_text": chosen_row["text"],
            "rejected_text": rejected_row["text"],
        }
    )

label_rows = []
for row in response_rows:
    if row["variant"] == "chosen":
        label = 1
    elif row["variant"] == "rejected":
        label = 0
    elif row["variant"] == "desirable_only":
        label = 1
    else:
        label = 0

    label_rows.append(
        {
            "response_id": row["response_id"],
            "prompt_id": row["prompt_id"],
            "split": row["split"],
            "prompt": row["prompt"],
            "label": label,
            "text": row["text"],
        }
    )

pair_df = pd.DataFrame(pair_rows)
label_df = pd.DataFrame(label_rows)

display(pair_df.head(8))
label_df.head(10)

## Step 4: Build tensors and initialize the SFT-style starting point

All three methods start from the same initial policy weights. DPO additionally keeps a frozen reference copy, while ORPO and KTO do not need one during optimization.

In [ ]:
feature_matrix = {
    row["response_id"]: torch.tensor(row["features"], dtype=torch.float32)
    for row in response_rows
}
response_lookup = {row["response_id"]: row for row in response_rows}

pair_train_rows = pair_df[pair_df["split"] == "train"].to_dict("records")
pair_eval_rows = pair_df[pair_df["split"] == "eval"].to_dict("records")
label_train_rows = label_df[label_df["split"] == "train"].to_dict("records")
label_eval_rows = label_df[label_df["split"] == "eval"].to_dict("records")

reference_weights = torch.tensor([1.04, 1.20, 1.48, 0.98, 0.82], dtype=torch.float32)
initial_weights = reference_weights.clone().detach()

pd.DataFrame(
    {
        "feature": feature_names,
        "reference_weight": reference_weights.tolist(),
        "initial_weight": initial_weights.tolist(),
    }
)

## Step 5: Implement tiny DPO, ORPO, and KTO objectives

These are score-level analogues of the full token-level objectives. They are intentionally simple so you can see the role of the reference model, the reference-free odds penalty, and binary desirable and undesirable supervision.

In [ ]:
def score_response(weight_vector, response_id):
    return torch.dot(weight_vector, feature_matrix[response_id])

def dpo_loss(weight_vector, reference_vector, chosen_id, rejected_id, beta=0.7):
    policy_gap = score_response(weight_vector, chosen_id) - score_response(weight_vector, rejected_id)
    reference_gap = score_response(reference_vector, chosen_id) - score_response(reference_vector, rejected_id)
    return -F.logsigmoid(beta * (policy_gap - reference_gap))

def orpo_loss(weight_vector, chosen_id, rejected_id, alpha=0.45):
    chosen_score = score_response(weight_vector, chosen_id)
    rejected_score = score_response(weight_vector, rejected_id)
    positive_term = F.softplus(-chosen_score)
    odds_ratio_term = F.softplus(-(chosen_score - rejected_score))
    return positive_term + alpha * odds_ratio_term

def kto_loss(weight_vector, response_id, label, bad_weight=1.2):
    score = score_response(weight_vector, response_id)
    target = torch.tensor(float(label), dtype=torch.float32)
    weight = bad_weight if label == 0 else 1.0
    return weight * F.binary_cross_entropy_with_logits(score, target)

def pair_accuracy(pair_records, weight_vector):
    wins = []
    for row in pair_records:
        chosen_score = float(score_response(weight_vector, row["chosen_id"]).detach())
        rejected_score = float(score_response(weight_vector, row["rejected_id"]).detach())
        wins.append(int(chosen_score >= rejected_score))
    return round(sum(wins) / len(wins), 4)

def label_accuracy(label_records, weight_vector):
    rows = []
    for row in label_records:
        probability = torch.sigmoid(score_response(weight_vector, row["response_id"])).item()
        prediction = 1 if probability >= 0.5 else 0
        rows.append(int(prediction == row["label"]))
    return round(sum(rows) / len(rows), 4)

print("Pair accuracy before training:", pair_accuracy(pair_train_rows, initial_weights))
print("Label accuracy before training:", label_accuracy(label_train_rows, initial_weights))

## Step 6: Train DPO, ORPO, and KTO side by side

The optimization loop is deliberately uniform so the results stay comparable. DPO uses pair data plus a reference. ORPO uses pair data without a reference. KTO uses only binary labels on individual responses.

In [ ]:
def train_model(kind, epochs=70, lr=0.08):
    weights = initial_weights.clone().detach().requires_grad_(True)
    optimizer = torch.optim.Adam([weights], lr=lr)
    history = []

    for epoch in range(1, epochs + 1):
        losses = []

        if kind in {"dpo", "orpo"}:
            shuffled_pairs = pair_train_rows.copy()
            random.shuffle(shuffled_pairs)
            for row in shuffled_pairs:
                if kind == "dpo":
                    loss = dpo_loss(weights, reference_weights, row["chosen_id"], row["rejected_id"])
                else:
                    loss = orpo_loss(weights, row["chosen_id"], row["rejected_id"])

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                losses.append(float(loss.detach()))
        else:
            shuffled_labels = label_train_rows.copy()
            random.shuffle(shuffled_labels)
            for row in shuffled_labels:
                loss = kto_loss(weights, row["response_id"], row["label"])
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
                losses.append(float(loss.detach()))

        history.append(
            {
                "epoch": epoch,
                "loss": round(sum(losses) / len(losses), 6),
                "train_pair_accuracy": pair_accuracy(pair_train_rows, weights),
                "eval_pair_accuracy": pair_accuracy(pair_eval_rows, weights),
                "train_label_accuracy": label_accuracy(label_train_rows, weights),
                "eval_label_accuracy": label_accuracy(label_eval_rows, weights),
            }
        )

    return weights.detach(), pd.DataFrame(history)

dpo_weights, dpo_history = train_model("dpo")
orpo_weights, orpo_history = train_model("orpo")
kto_weights, kto_history = train_model("kto")

pd.DataFrame(
    [
        {"method": "DPO", "final_train_pair_accuracy": dpo_history.iloc[-1]["train_pair_accuracy"], "final_eval_pair_accuracy": dpo_history.iloc[-1]["eval_pair_accuracy"]},
        {"method": "ORPO", "final_train_pair_accuracy": orpo_history.iloc[-1]["train_pair_accuracy"], "final_eval_pair_accuracy": orpo_history.iloc[-1]["eval_pair_accuracy"]},
        {"method": "KTO", "final_train_pair_accuracy": kto_history.iloc[-1]["train_pair_accuracy"], "final_eval_pair_accuracy": kto_history.iloc[-1]["eval_pair_accuracy"]},
    ]
)

## Step 7: Plot the training logs

Loss curves are useful, but the more practical view is whether each method improves the metric that matches its data format. DPO and ORPO should raise pair accuracy. KTO should raise label accuracy and still improve pair ranking if the labels are informative.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(dpo_history["epoch"], dpo_history["loss"], label="DPO", color="#4C72B0")
axes[0].plot(orpo_history["epoch"], orpo_history["loss"], label="ORPO", color="#55A868")
axes[0].plot(kto_history["epoch"], kto_history["loss"], label="KTO", color="#C44E52")
axes[0].set_title("Training loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(dpo_history["epoch"], dpo_history["eval_pair_accuracy"], label="DPO", color="#4C72B0")
axes[1].plot(orpo_history["epoch"], orpo_history["eval_pair_accuracy"], label="ORPO", color="#55A868")
axes[1].plot(kto_history["epoch"], kto_history["eval_pair_accuracy"], label="KTO", color="#C44E52")
axes[1].set_title("Held-out pair accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

axes[2].plot(dpo_history["epoch"], dpo_history["eval_label_accuracy"], label="DPO", color="#4C72B0")
axes[2].plot(orpo_history["epoch"], orpo_history["eval_label_accuracy"], label="ORPO", color="#55A868")
axes[2].plot(kto_history["epoch"], kto_history["eval_label_accuracy"], label="KTO", color="#C44E52")
axes[2].set_title("Held-out label accuracy")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Accuracy")
axes[2].legend()

plt.tight_layout()
plt.show()

## Step 8: Compare the final evaluation metrics

This is the practical comparison view. DPO is strongest when you have good pairs and a trustworthy SFT reference. ORPO trades the reference away. KTO is most natural when your feedback arrives as binary labels.

In [ ]:
final_metrics_df = pd.DataFrame(
    [
        {
            "method": "DPO",
            "needs_reference": True,
            "train_pair_accuracy": dpo_history.iloc[-1]["train_pair_accuracy"],
            "eval_pair_accuracy": dpo_history.iloc[-1]["eval_pair_accuracy"],
            "train_label_accuracy": dpo_history.iloc[-1]["train_label_accuracy"],
            "eval_label_accuracy": dpo_history.iloc[-1]["eval_label_accuracy"],
        },
        {
            "method": "ORPO",
            "needs_reference": False,
            "train_pair_accuracy": orpo_history.iloc[-1]["train_pair_accuracy"],
            "eval_pair_accuracy": orpo_history.iloc[-1]["eval_pair_accuracy"],
            "train_label_accuracy": orpo_history.iloc[-1]["train_label_accuracy"],
            "eval_label_accuracy": orpo_history.iloc[-1]["eval_label_accuracy"],
        },
        {
            "method": "KTO",
            "needs_reference": False,
            "train_pair_accuracy": kto_history.iloc[-1]["train_pair_accuracy"],
            "eval_pair_accuracy": kto_history.iloc[-1]["eval_pair_accuracy"],
            "train_label_accuracy": kto_history.iloc[-1]["train_label_accuracy"],
            "eval_label_accuracy": kto_history.iloc[-1]["eval_label_accuracy"],
        },
    ]
)

final_metrics_df

## Step 9: Compare concrete prompt choices across methods

Metrics are easier to trust when they line up with visible prompt-level behavior. The table below shows which response each method prefers on held-out prompts.

In [ ]:
def winner_for_pair(weight_vector, pair_record):
    chosen_score = float(score_response(weight_vector, pair_record["chosen_id"]).detach())
    rejected_score = float(score_response(weight_vector, pair_record["rejected_id"]).detach())
    winner_id = pair_record["chosen_id"] if chosen_score >= rejected_score else pair_record["rejected_id"]
    return winner_id, round(chosen_score - rejected_score, 4)

comparison_rows = []
for row in pair_eval_rows:
    dpo_winner, dpo_margin = winner_for_pair(dpo_weights, row)
    orpo_winner, orpo_margin = winner_for_pair(orpo_weights, row)
    kto_winner, kto_margin = winner_for_pair(kto_weights, row)

    comparison_rows.append(
        {
            "prompt_id": row["prompt_id"],
            "prompt": row["prompt"],
            "dpo_winner": response_lookup[dpo_winner]["variant"],
            "orpo_winner": response_lookup[orpo_winner]["variant"],
            "kto_winner": response_lookup[kto_winner]["variant"],
            "dpo_margin": dpo_margin,
            "orpo_margin": orpo_margin,
            "kto_margin": kto_margin,
        }
    )

comparison_df = pd.DataFrame(comparison_rows)
comparison_df

## Step 10: When ORPO is usually preferable

ORPO is most attractive when you still have pairwise data but do not want to keep a separate frozen reference model in memory or in the training loop. It is a particularly practical choice when infrastructure simplicity matters.

In [ ]:
orpo_scenarios_df = pd.DataFrame(
    [
        {
            "signal": "You have good chosen and rejected pairs.",
            "why_orpo_helps": "It keeps pairwise supervision but removes the extra reference pass.",
            "watch_out": "Pair quality still matters just as much as it does for DPO.",
        },
        {
            "signal": "GPU memory is tight.",
            "why_orpo_helps": "Reference-free training reduces bookkeeping and memory pressure.",
            "watch_out": "You still need to monitor regressions because there is no frozen reference anchor.",
        },
        {
            "signal": "The SFT checkpoint is not stable enough to trust as a reference.",
            "why_orpo_helps": "A reference-free objective avoids baking that checkpoint directly into the loss.",
            "watch_out": "Without a good evaluation loop, reference-free alignment can drift in style.",
        },
    ]
)

orpo_scenarios_df

## Step 11: When KTO is usually preferable

KTO is a better fit when the easiest feedback to collect is binary approval or disapproval rather than fully paired rankings. Production thumbs-up and thumbs-down data often looks much closer to KTO supervision than to DPO data.

In [ ]:
kto_scenarios_df = pd.DataFrame(
    [
        {
            "signal": "Feedback arrives as thumbs-up and thumbs-down events.",
            "why_kto_helps": "You can train directly on binary desirable and undesirable labels.",
            "watch_out": "Binary labels lose some of the fine-grained information that full rankings provide.",
        },
        {
            "signal": "You have many standalone good and bad examples but few explicit pairs.",
            "why_kto_helps": "KTO can use those examples without constructing synthetic pairs.",
            "watch_out": "Label imbalance can bias the model if negatives dominate.",
        },
        {
            "signal": "You want alignment data from real product logs.",
            "why_kto_helps": "Binary product feedback is often easier to collect than adjudicated preference pairs.",
            "watch_out": "Implicit user feedback can be noisy and need filtering.",
        },
    ]
)

kto_scenarios_df

## Step 12: Stress test weak-reference and label-only situations

This small stress test shows why alternatives exist. If the reference is weak, DPO can inherit that weakness. If pair data disappears entirely, KTO still has a workable supervision path.

In [ ]:
weak_reference = reference_weights + torch.tensor([-0.20, -0.12, -0.42, -0.08, 0.04], dtype=torch.float32)

def train_dpo_with_reference(custom_reference, epochs=40):
    weights = initial_weights.clone().detach().requires_grad_(True)
    optimizer = torch.optim.Adam([weights], lr=0.08)
    for _ in range(epochs):
        shuffled_pairs = pair_train_rows.copy()
        random.shuffle(shuffled_pairs)
        for row in shuffled_pairs:
            loss = dpo_loss(weights, custom_reference, row["chosen_id"], row["rejected_id"])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return weights.detach()

weak_dpo_weights = train_dpo_with_reference(weak_reference)

label_only_rows = [row for row in label_train_rows if row["prompt_id"] in {"p01", "p02", "p03", "p09", "p10"}]

def train_kto_label_only(custom_rows, epochs=50):
    weights = initial_weights.clone().detach().requires_grad_(True)
    optimizer = torch.optim.Adam([weights], lr=0.08)
    for _ in range(epochs):
        shuffled_rows = custom_rows.copy()
        random.shuffle(shuffled_rows)
        for row in shuffled_rows:
            loss = kto_loss(weights, row["response_id"], row["label"])
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
    return weights.detach()

label_only_kto_weights = train_kto_label_only(label_only_rows)

stress_test_df = pd.DataFrame(
    [
        {
            "scenario": "Standard DPO reference",
            "eval_pair_accuracy": pair_accuracy(pair_eval_rows, dpo_weights),
            "eval_label_accuracy": label_accuracy(label_eval_rows, dpo_weights),
        },
        {
            "scenario": "Weak-reference DPO",
            "eval_pair_accuracy": pair_accuracy(pair_eval_rows, weak_dpo_weights),
            "eval_label_accuracy": label_accuracy(label_eval_rows, weak_dpo_weights),
        },
        {
            "scenario": "ORPO without reference",
            "eval_pair_accuracy": pair_accuracy(pair_eval_rows, orpo_weights),
            "eval_label_accuracy": label_accuracy(label_eval_rows, orpo_weights),
        },
        {
            "scenario": "KTO from label-only subset",
            "eval_pair_accuracy": pair_accuracy(pair_eval_rows, label_only_kto_weights),
            "eval_label_accuracy": label_accuracy(label_eval_rows, label_only_kto_weights),
        },
    ]
)

stress_test_df

## Step 13: Save artifacts and build a practical decision matrix

The notebook ends with a reusable summary: metrics, history files, and a compact decision matrix that compares when DPO, ORPO, and KTO are the right tool.

In [ ]:
decision_matrix_df = pd.DataFrame(
    [
        {
            "method": "DPO",
            "supervision_shape": "chosen and rejected pairs",
            "needs_reference": "yes",
            "best_when": "you have strong SFT and high-quality preference pairs",
            "main_risk": "a poor reference or noisy pairs can distort the update",
        },
        {
            "method": "ORPO",
            "supervision_shape": "chosen and rejected pairs",
            "needs_reference": "no",
            "best_when": "you want pairwise alignment without an extra reference model",
            "main_risk": "without a strong evaluation loop, style drift can hide inside good pair metrics",
        },
        {
            "method": "KTO",
            "supervision_shape": "binary desirable and undesirable labels",
            "needs_reference": "no",
            "best_when": "feedback arrives as thumbs-up and thumbs-down or standalone examples",
            "main_risk": "binary labels can be noisier and less informative than full rankings",
        },
    ]
)

final_metrics_path = ARTIFACT_DIR / "alignment_comparison_metrics.json"
history_path = ARTIFACT_DIR / "alignment_histories.json"

with final_metrics_path.open("w", encoding="utf-8") as handle:
    json.dump(final_metrics_df.to_dict("records"), handle, indent=2)

with history_path.open("w", encoding="utf-8") as handle:
    json.dump(
        {
            "dpo": dpo_history.to_dict("records"),
            "orpo": orpo_history.to_dict("records"),
            "kto": kto_history.to_dict("records"),
        },
        handle,
        indent=2,
    )

print("Saved:", final_metrics_path)
print("Saved:", history_path)
decision_matrix_df

## Key takeaways

- DPO is strongest when you have reliable pairwise preferences and a trustworthy SFT reference.
- ORPO keeps the pairwise formulation but removes the reference model from the optimization loop.
- KTO is especially useful when alignment feedback arrives as binary desirable and undesirable labels.
- The right objective depends on the data you can collect and the training setup you can reliably run, not on fashion alone.